<a href="https://colab.research.google.com/github/Auta01/Pytorch/blob/main/pytorch_computer_vision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
### importing computer vision libaries
import torch
from torch import nn
import torchvision
from torchvision import datasets
from torchvision import transforms
from torchvision.transforms import ToTensor


import matplotlib.pyplot as plt
!pip install torchvision

In [ ]:
##Getting a dataset
#setup training data
from torchvision import datasets
train_data = datasets.FashionMNIST(
    root='data',
    train=True,
    download=True,
    transform=torchvision.transforms.ToTensor(),
    target_transform=None
)

test_data =datasets.FashionMNIST(
    root = 'data',
    train=False,
    download=True,
    transform=torchvision.transforms.ToTensor(),
    target_transform=None
)

In [ ]:
len(train_data), len(test_data)

In [ ]:
class_names =train_data.classes
class_names

In [ ]:
class_to_idx = train_data.class_to_idx
class_to_idx

In [ ]:
train_data.targets

In [ ]:
#check input and outputshaes of data
image, label = train_data[0]
print(f'Image shape:{image.shape} -> [color_channels, height, width]')
print(f'Image label:{class_names[label]}')

In [ ]:
#visualizing our images
import matplotlib.pyplot as plt
plt.imshow(image.squeeze())
plt.title(label);

In [ ]:
plt.imshow(image.squeeze(), cmap='gray')
plt.title(class_names[label]);
plt.axis(False);

In [ ]:
#plot more images
torch.manual_seed(42)
fig = plt.figure(figsize=(9,9))
rows, cols = 4 ,4
for i in range (1, rows*cols+1):
  random_idx = torch.randint(0, len(train_data), size=[1]).item()
  img, laabel  =train_data[random_idx]
  fig.add_subplot(rows, cols, i)
  plt.imshow(img.squeeze(), cmap='gray')
  plt.title(class_names[laabel]); # Corrected from 'label' to 'laabel'
  plt.axis(False) ;

In [ ]:
train_data,test_data

In [ ]:
#prepare dataloader
from torch.utils.data import DataLoader
BATCH_SIZE = 32
train_dataloader = DataLoader(dataset=train_data,
                              batch_size=BATCH_SIZE,
                              shuffle=True)

test_dataloader = DataLoader(dataset=test_data,
                             batch_size =BATCH_SIZE,
                             shuffle=False)
train_dataloader, test_dataloader

In [ ]:
#lets check out what we have create
print(f'DataLoaders:{train_dataloader, test_dataloader}')
print(f'length of train_dataloader:{len({train_dataloader})}batch of{BATCH_SIZE}')
print(f'length of test_dataloader: {len({test_dataloader})}batch of{BATCH_SIZE}')

In [ ]:
#checj out what inside the trainig dataloader
train_feature_batch, train_labels_batch = next(iter(train_dataloader))
train_feature_batch.shape, train_labels_batch.shape

In [ ]:
#show a sample
#torch.manual_seed(42)
random_idx = torch.randint(0, len(train_feature_batch), size=[1]).item()
img, label = train_feature_batch[random_idx], train_labels_batch[random_idx]
plt.imshow(img.squeeze(), cmap='gray')
plt.title(class_names[label])
plt.axis(False)


In [ ]:
#Building a baseline model
#create a layer
flatten_model =nn.Flatten()

#get  a single sample
x =train_feature_batch[0]

#Flatten the samole
output =flatten_model(x)
#print out what happen
print(f'Shape before flattening:{x.shape}->[color_channels, height, width]')
print(f'Shape after flattening:{output.shape}->[color_channels, height*width]')


In [ ]:
from torch import nn
class FashionMNISTV0(nn.Module):
  def __init__(self,
               input_shape:int,
               hidden_units:int,
               output_shape:int):
    super().__init__()
    self.layer_stack = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features= input_shape,
                  out_features = hidden_units),
        nn.Linear(in_features =hidden_units,
                  out_features=output_shape)
    )

  def forward (self, x):
    return self.layer_stack(x)

In [ ]:
torch.manual_seed(42)

model_0= FashionMNISTV0(
    input_shape=784,
    hidden_units=10,
    output_shape =len(class_names)

  )
model_0

In [ ]:
dummy_x = torch.rand([1,1,28,28])

model_0(dummy_x)

In [ ]:
#Creating a loss function, optimizer and evaluation metrics
import requests
from pathlib import Path

#Download helper_function from pytorch repo
if Path('helper_function.py').is_file():
  print('helper_function.py already exists')

else:
  print('Downloading helper_function.py')
requests = requests.get('https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/refs/heads/main/helper_functions.py')
with open ('helper_function.py','wb')as f:
 f.write(requests.content)



In [ ]:
#importaccuracy metrics
from helper_function import accuracy_fn
#set up loss function and optimzeer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params =model_0.parameters(),
                            lr=0.01)

In [ ]:
#Creating a function to time experiments
from timeit import default_timer as timer
def print_train_time(start: float,
                     end: float,
                     device: torch.device=None):
  total_time =end -start
  print(f'Train time on{device}:{total_time:.3f}seconds')
  return total_time


In [ ]:
start_time =timer()

end_time =timer()
print_train_time(start =start_time, end= end_time, device='cpu')

In [ ]:
#Creating a training loop and training a model
# import tqdm for progress bar
from tqdm.auto import tqdm
torch.manual_seed(42)
train_time_start_on_cpu = timer()

epoch =3
for epoch in tqdm(range(epoch)):
  print(f'Epoch: {epoch}')
  ##Training
  train_loss = 0
  for batch, (x,y) in enumerate(train_dataloader):
    model_0.train()
    #forward pass
    y_pred =model_0(x)
    #calculate loss per batch
    loss = loss_fn(y_pred, y)
    train_loss += loss
    #optimiser zero grad
    optimizer.zero_grad()
    #loss backward
    loss.backward()
    #optimizer step
    optimizer.step()

    #print out what is happenin
    if batch % 400 ==0:
      print(f'Looked at{batch*len(x)}/{len(train_dataloader.dataset)}')


  #Divide total train modelby length of train dataloader
  train_loss /= len(train_dataloader)

  ##Testing
  test_loss, test_acc= 0,0

  model_0.eval()
  with torch.inference_mode():
    for x_test, y_test in test_dataloader:
      #Forward pass
      test_pred = model_0(x_test)
      #calculate loss(accumulatively)
      test_loss += loss_fn(test_pred, y_test)
      #calculate the accuracy
      test_acc += accuracy_fn(y_true=y_test,y_pred= test_pred.argmax(dim=1))
      #calculate thr test loss average per bacth
    test_loss /= len(test_dataloader)
    #calculate the acc average per batch
    test_acc /= len(test_dataloader)
    #print what is happening
    print(f'Train loss:{train_loss:.4f}|Test loss:{test_loss:.4f},Test acc:{test_acc:.4f}')
#calculate training time
train_time_end_on_cpu = timer()

total_train_time_model_0 = print_train_time(start =train_time_start_on_cpu,
                                            end=train_time_end_on_cpu,
                                            device=str(next(model_0.parameters()).device))

In [ ]:
#Make prediction and get model results
torch.manual_seed(42)
def eval_model(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn:torch.nn.Module,
               accuracy_fn):
  loss, acc = 0.0, 0.0
  model.eval()
  with torch.inference_mode():
    for x ,y in tqdm (data_loader):
      #Make prediction
      y_pred = model(x)
      #Accumulate the loss and acc values per batch
      loss += loss_fn(y_pred, y)
      acc += accuracy_fn(y_true=y, y_pred=y_pred.argmax(dim=1))

    # Divide total loss and acc by length of data_loader
    loss /= len(data_loader)
    acc /= len(data_loader)

  return {'model_name': model.__class__.__name__,
          'model_loss': loss.item(),
          'model_acc': acc}

#Calculate model 0 result on test dataset
model_0_results = eval_model(model=model_0,
                             data_loader=test_dataloader,
                             loss_fn=loss_fn,
                             accuracy_fn=accuracy_fn)

model_0_results

In [ ]:
#setuo  a device agnostic code
torch.cuda.is_available()

In [ ]:
!nvidia-smi

In [ ]:
#set device agnostic code
import torch
device ='cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
#Creating a model non-linearity
class FashionMNISTV1(nn.Module):
  def __init__(self,
               input_shape: int,
               hidden_units: int,
               output_shape: int):
    super().__init__()
    self.layer_stack = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features= input_shape,
                  out_features= hidden_units),
        nn.ReLU(),
        nn.Linear(in_features =hidden_units,
               out_features =output_shape),
        nn.ReLU()
    )

  def forward(self, x: torch.Tensor):
    return self.layer_stack(x)

In [ ]:
#Create an instance of model_1

torch.manual_seed(42)
model_1 = FashionMNISTV1(input_shape=784,
                         hidden_units=10,
                         output_shape=len(class_names)).to(device)

In [ ]:
##Setup loss and optimizer
from helper_function import accuracy_fn
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params =model_1.parameters(),
                            lr=0.1)

In [ ]:
#Training and evaluation /testing loops
#Creating a training loop and training

def train_step(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               accuracy_fn,
               device: torch.device = device):

   # Put model into training mode
   model.train()
   train_loss, train_acc = 0, 0 # Initialize for the current epoch

   # Loop over batches
   for batch, (x,y) in enumerate(data_loader):
    # Put data on target device
    x, y = x.to(device),  y.to(device)

    # 1. Forward pass
    y_pred = model(x) # Use 'model' from function argument

    # 2. Calculate loss and accuracy
    loss = loss_fn(y_pred, y)
    train_loss += loss.item() # Accumulate scalar loss value
    train_acc += accuracy_fn(y_true=y, y_pred=y_pred.argmax(dim=1))

    # 3. Optimizer zero grad
    optimizer.zero_grad()

    # 4. Loss backward
    loss.backward()

    # 5. Optimizer step
    optimizer.step()

    # Print out what is happening
    if batch % 400 == 0:
      print(f'Looked at {batch*len(x)}/{len(data_loader.dataset)} samples')

   # Adjust metrics to get average loss and accuracy per batch
   train_loss /= len(data_loader)
   train_acc /= len(data_loader)

   return train_loss, train_acc # Return for logging/tracking


In [ ]:
def test_step(model: torch.nn.Module,
              data_loader:torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              accuracy_fn,
              device: torch.device = device):
  test_loss, test_acc = 0,0

  model.eval()
  #Turn an inference mode and
  with torch.inference_mode():
    for x, y in data_loader:
      x,y = x.to(device),y.to(device)

      #forward pass
      test_pred = model(x)

      #calculate the loss/acc
      test_loss += loss_fn(test_pred, y)
      test_acc += accuracy_fn(y_true=y, y_pred =test_pred.argmax(dim=1))

    #Adjust
    test_loss /= len(data_loader)
    test_acc /=len(data_loader)
    print(f'Test loss:{test_loss:.3f}|Test acc:{test_acc:.2f}')

In [ ]:
torch.manual_seed(42)

from timeit import default_timer as timer
train_time_start_on_cpu = timer()

#set epoch
epoch = 3

#Create a optimization and evaluation loop using train_step() and test_step()
for epoch in tqdm(range(epoch)):
  print(f'Epoch: {epoch}')
  train_step(model= model_1,
             data_loader = train_dataloader,
             loss_fn= loss_fn,
             optimizer = optimizer,
             accuracy_fn = accuracy_fn,
             device =device)

  test_step(model =model_1,
            data_loader = test_dataloader,
            loss_fn =loss_fn,
            accuracy_fn = accuracy_fn,
            device = device)


  train_time_end_on_cpu = timer()
  total_train_time_model_1 =print_train_time(start =train_time_start_on_cpu,
                                             end =train_time_end_on_cpu,
                                             device=device)

In [ ]:
model_0_results

In [ ]:
#Train time on cpu
total_train_time_model_0

In [ ]:
import torch.nn as nn
#Building a convolutional neural network
class FashionMNISTV2(nn.Module):
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
        super().__init__()
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=input_shape,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_units,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units,
                         out_channels=hidden_units,
                         kernel_size=3,
                         stride=1,
                         padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=hidden_units,
                         out_features=output_shape)
        )

    def forward(self , x):
        x = self.conv_block_1(x)
        # print(x.shape) # Removed print statements for cleaner execution

        x = self.conv_block_2(x)
        # print(x.shape) # Removed print statements for cleaner execution
        x = self.classifier(x)
        return x

In [ ]:
pip install notebook

In [ ]:
model_2 = FashionMNISTV2(input_shape=1,
                         hidden_units=10,
                         output_shape=len(class_names)).to(device)

In [ ]:
##stepping throungh nn.conv2d
torch.manual_seed(42)

#create a batch of images

images = torch.randn(size=(32, 3, 64,64))
test_image = images[0]

print(f'image batch shape:{image.shape}')
print(f'sinlge image shape: {test_image.shape}')
print(f'Test image {test_image}')

In [ ]:
model_2.state_dict()

In [ ]:
conv_layer = nn.Conv2d(in_channels=1,
                       out_channels =10,
                       kernel_size=3,
                       stride=1,
                       padding=1)